# Step 9: Comprehensive Final Evaluation, Diagnostic Experiments & Production Serialization
**Project Title:** A Cost-Sensitive Machine Learning Framework for Optimizing Reverse Logistics Decisions in E-Commerce Return Management  
**Degree:** MSc Data Science - University of Wolverhampton (7CS043/7CS041)  

In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, precision_recall_curve

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11

print('======================================================================')
print('PHASE 1: LOADING BENCHMARK CSVs & COMPILING MASTER MATRIX')
print('======================================================================')

# Load exact benchmark CSV files from current directory
df_rf = pd.read_csv('baseline_results.csv')
print('Step 7 Random Forest Benchmark Results (baseline_results.csv) uploaded')
df_xgb = pd.read_csv('cost_sensitive_results.csv')
print('Step 8 XGBoost Benchmark Results (cost_sensitive_results.csv) uploaded')

# Master 6-Model Comparison Matrix
master_df = pd.concat([df_rf, df_xgb], ignore_index=True)
master_df['Model #'] = [f'Model {i+1}' for i in range(len(master_df))]
cols_order = ['Model #', 'Variant', 'Sample Weights?', 'Threshold Strategy', 'Threshold (t)', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC', 'Total Loss (R$)', 'Loss Per Order']
master_df = master_df[[c for c in cols_order if c in master_df.columns]]

print('\n======================================================================')
print('MASTER 6-MODEL COMPARISON MATRIX')
print('======================================================================')
print(master_df.to_string(index=False))

print('\n======================================================================')
print('PHASE 2: WINNING MODEL SELECTION & DETAILED COMPARATIVE RATIONALE')
print('======================================================================')

# Select winning model (Model 6: Cost-Aware XGBoost + OOF Threshold)
winner_row = master_df.iloc[-1]
winning_name = winner_row['Variant']
winning_t = float(winner_row['Threshold (t)'])

print(f'\n ABSOLUTE WINNING MODEL SELECTED: {winning_name}')
print(f'   Threshold (t)         : {winning_t:.2f}')
print(f'   Financial Loss / Order: {winner_row["Loss Per Order"]}')
print(f'   AUC-ROC               : {winner_row["AUC-ROC"]}')

# Load Master Dataset & Fit Winning Model
data_path = 'data_with_cost_matrix.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'price', 'freight_value', 'total_order_cost', 'shipping_cost_ratio',
    'return_shipping_cost_est', 'potential_loss', 'is_shipping_more_than_item',
    'freight_to_price_ratio', 'product_weight_g', 'product_length_cm', 
    'product_height_cm', 'product_width_cm', 'product_volume_cm3', 
    'product_photos_qty', 'density_g_cm3', 'delivery_delay_days',
    'customer_order_count', 'customer_avg_review', 'customer_return_rate', 
    'customer_total_spend', 'is_extreme_reviewer', 'reviewer_deviance_score',
    'product_return_rate', 'product_total_sales', 'category_return_rate',
    'haversine_distance_km'
]

le = LabelEncoder()
if 'product_category_name_english' in df.columns:
    df['category_encoded'] = le.fit_transform(df['product_category_name_english'].fillna('unknown'))
    feature_cols.append('category_encoded')

feature_cols = [c for c in feature_cols if c in df.columns]

X, y, w = df[feature_cols], df['is_returned'], df['sample_cost_weight']
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=0.20,shuffle=False)
test_indices = X_test.index
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'   scale_pos_weight               : {scale_pos_weight}')

winning_model = XGBClassifier(n_estimators=300, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, reg_alpha=0.1, reg_lambda=1.0, random_state=42, eval_metric='logloss')
winning_model.fit(X_train, y_train, sample_weight=w_train)
y_prob_win = winning_model.predict_proba(X_test)[:, 1]
y_pred_win = (y_prob_win >= winning_t).astype(int)


PHASE 1: LOADING BENCHMARK CSVs & COMPILING MASTER MATRIX
Step 7 Random Forest Benchmark Results (baseline_results.csv) uploaded
Step 8 XGBoost Benchmark Results (cost_sensitive_results.csv) uploaded

MASTER 6-MODEL COMPARISON MATRIX
Model #                                       Variant Sample Weights? Threshold Strategy  Threshold (t) Accuracy Precision Recall F1-Score AUC-ROC Total Loss (R$) Loss Per Order
Model 1                    Variant 1: Cost-Unaware RF              No   Default (t=0.50)           0.50   90.96%    80.69% 34.49%   48.32%  76.50%   R$ 103,299.86      R$ 4.6641
Model 2                      Variant 2: Cost-Aware RF             Yes   Default (t=0.50)           0.50   90.91%    79.78% 34.60%   48.27%  75.63%   R$ 102,796.21      R$ 4.6413
Model 3      Variant 3: Cost-Aware RF + OOF Threshold             Yes OOF Tuned (t=0.31)           0.31   89.85%    63.86% 39.65%   48.92%  75.63%   R$ 103,480.07      R$ 4.6722
Model 4               Variant 1: Cost-Unaware XGBoost 